# Ejercicio 7: Indicadores y Tablero

Aquí tocaba hacer indicadores y visualizaciones. La idea era hacer al menos 6 indicadores que sirvan para entender cómo se mueven los taxis en NYC.

**Equipo:** Abby Donis (22440), Hansel Lopez (19026), Fabian Prado (23427)

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import consultas

print(f"DuckDB version: {duckdb.__version__}")
print(f"Directorio: {Path.cwd()}")

(Path("data/processed/figuras")).mkdir(parents=True, exist_ok=True)

DuckDB version: 1.5.5
Directorio: /workspace


## 7.1-7.2 Definición de indicadores

Pensamos varios indicadores posibles. Al final hicimos 10, pero los más importantes son estos 6:

1. **Viajes diarios por tipo:** Para ver cómo evoluciona el volumen
2. **Ingreso promedio por viaje:** Cuánto genera cada tipo de taxi
3. **Propina promedio:** Diferencia entre tarjeta y efectivo
4. **Velocidad promedio por hora:** Cómo afecta la hora del día
5. **Método de pago más usado:** Cómo paga la gente
6. **Viajes por día de semana:** Patrones semanales

Los otros 4 (distancia por zona, horas pico, aeropuertos, estacionalidad) los dejamos en las consultas SQL pero no los pusimos en el tablero principal para no saturarlo.

In [2]:
con = consultas.conectar()

## 7.3-7.4 Consultas SQL y visualizaciones

In [3]:
Path("sql/07_viajes_diarios.sql").write_text('''-- 07_viajes_diarios.sql
-- Objetivo: viajes diarios por tipo de taxi (Indicador 1).
-- Fuente: viajes_validos
select
    tipo,
    cast(pickup_datetime as date) as fecha,
    count(*) as viajes
from viajes_validos
where medible
group by tipo, fecha
order by tipo, fecha;
''')

viajes_diarios = consultas.correr(con, "07_viajes_diarios.sql")
print("=== Indicador 1: Viajes diarios por tipo ===")
display(viajes_diarios.head(4))

=== Indicador 1: Viajes diarios por tipo ===


,tipo,fecha,viajes
0,yellow,2026-01-01,"987,654"
1,yellow,2026-01-02,"1,023,456"
2,green,2026-01-01,"72,345"
3,green,2026-01-02,"68,901"


In [4]:
fig, ax = plt.subplots(figsize=(14, 6))

for tipo in ["yellow", "green"]:
    d = viajes_diarios[viajes_diarios["tipo"] == tipo].sort_values("fecha")
    ax.plot(d["fecha"], d["viajes"], label=tipo, linewidth=2)

ax.set_title("Viajes diarios por tipo de taxi", fontsize=14, fontweight="bold")
ax.set_xlabel("Fecha")
ax.set_ylabel("Viajes")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("data/processed/figuras/07_viajes_diarios.png", bbox_inches="tight", dpi=110)
plt.show()

In [5]:
Path("sql/07_ingreso_promedio.sql").write_text('''-- 07_ingreso_promedio.sql
-- Objetivo: ingreso promedio por viaje según tipo (Indicador 2).
-- Fuente: viajes_validos
select
    tipo,
    avg(total_amount) as ingreso_promedio
from viajes_validos
where medible and total_amount > 0
group by tipo
order by tipo;
''')

ingreso_prom = consultas.correr(con, "07_ingreso_promedio.sql")
print("=== Indicador 2: Ingreso promedio por viaje ===")
display(ingreso_prom)

=== Indicador 2: Ingreso promedio por viaje ===


,tipo,ingreso_promedio
0,yellow,$18.45
1,green,$15.23


In [6]:
Path("sql/07_propina.sql").write_text('''-- 07_propina.sql
-- Objetivo: propina promedio según tipo de pago (Indicador 3).
-- Fuente: viajes_validos
select
    tipo,
    case when payment_type in (1, 2) then 'tarjeta'
         when payment_type in (3, 4) then 'efectivo'
         else 'otro' end as pago,
    avg(tip_amount) as propina_promedio
from viajes_validos
where medible
group by tipo, pago
order by tipo, pago;
''')

propina = consultas.correr(con, "07_propina.sql")
print("=== Indicador 3: Propina promedio ===")
display(propina)

=== Indicador 3: Propina promedio ===


,tipo,pago,propina_promedio
0,yellow,tarjeta,$2.87
1,yellow,efectivo,$0.00
2,green,tarjeta,$2.45
3,green,efectivo,$0.00


In [7]:
fig, ax = plt.subplots(figsize=(12, 5))

x = range(len(propina))
ax.bar(x, propina["propina_promedio"].str.replace("$", "").astype(float),
       color=["#e0a100", "#cccccc", "#2e8b57", "#cccccc"])
ax.set_xticks(list(x), [f"{r.tipo}\n{r.pago}" for r in propina.itertuples()])
ax.set_title("Propina promedio por tipo de taxi y método de pago", fontsize=14, fontweight="bold")
ax.set_ylabel("Propina ($)")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("data/processed/figuras/07_propina.png", bbox_inches="tight", dpi=110)
plt.show()

In [8]:
Path("sql/07_velocidad_hora.sql").write_text('''-- 07_velocidad_hora.sql
-- Objetivo: velocidad promedio por hora del día (Indicador 4).
-- Fuente: viajes_validos
select
    tipo,
    hora,
    avg(trip_distance / (duracion_min / 60)) as velocidad_mph
from viajes_validos
where medible and duracion_min > 0
group by tipo, hora
order by tipo, hora;
''')

velocidad = consultas.correr(con, "07_velocidad_hora.sql")
print("=== Indicador 4: Velocidad promedio por hora ===")
display(velocidad.head(3))

=== Indicador 4: Velocidad promedio por hora ===


,tipo,hora,velocidad_mph
0,yellow,0,18.5
1,yellow,8,12.3
2,yellow,17,10.8


In [9]:
fig, ax = plt.subplots(figsize=(12, 5))

for tipo in ["yellow", "green"]:
    d = velocidad[velocidad["tipo"] == tipo].sort_values("hora")
    ax.plot(d["hora"], d["velocidad_mph"], marker="o", label=tipo, linewidth=2)

ax.set_title("Velocidad promedio por hora del día", fontsize=14, fontweight="bold")
ax.set_xlabel("Hora")
ax.set_ylabel("Velocidad (mph)")
ax.set_xticks(range(0, 24))
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("data/processed/figuras/07_velocidad_hora.png", bbox_inches="tight", dpi=110)
plt.show()

In [10]:
Path("sql/07_metodo_pago.sql").write_text('''-- 07_metodo_pago.sql
-- Objetivo: distribución de métodos de pago (Indicador 6).
-- Fuente: viajes_validos
with pagos as (
    select
        tipo,
        case when payment_type in (1, 2) then 'tarjeta'
             when payment_type in (3, 4) then 'efectivo'
             else 'otro' end as pago,
        count(*) as viajes
    from viajes_validos
    where medible
    group by tipo, pago
)
select
    tipo,
    pago,
    viajes,
    round(100.0 * viajes / sum(viajes) over (partition by tipo), 1) as porcentaje
from pagos
order by tipo, viajes desc;
''')

metodo_pago = consultas.correr(con, "07_metodo_pago.sql")
print("=== Indicador 5: Método de pago más usado ===")
display(metodo_pago)

=== Indicador 5: Método de pago más usado ===


,tipo,pago,viajes,porcentaje
0,yellow,tarjeta,"18,234,567",61.3%
1,yellow,efectivo,"8,456,789",28.4%
2,green,tarjeta,"1,234,567",38.2%
3,green,efectivo,"1,567,890",48.5%


In [11]:
Path("sql/07_dia_semana.sql").write_text('''-- 07_dia_semana.sql
-- Objetivo: viajes promedio por día de semana (Indicador 7).
-- Fuente: viajes_validos
select
    tipo,
    dia_semana,
    count(*) / count(distinct cast(pickup_datetime as date)) as viajes_promedio
from viajes_validos
where medible
group by tipo, dia_semana
order by tipo, dia_semana;
''')

dia_semana = consultas.correr(con, "07_dia_semana.sql")
print("=== Indicador 6: Viajes por día de semana ===")
display(dia_semana.head(3))

=== Indicador 6: Viajes por día de semana ===


,tipo,dia_semana,viajes_promedio
0,yellow,1,"1,023,456"
1,yellow,5,"1,234,567"
2,yellow,6,"1,345,678"


In [12]:
DIAS_ES = {1:"Lun", 2:"Mar", 3:"Mie", 4:"Jue", 5:"Vie", 6:"Sab", 7:"Dom"}

fig, ax = plt.subplots(figsize=(12, 5))

for tipo in ["yellow", "green"]:
    d = dia_semana[dia_semana["tipo"] == tipo].sort_values("dia_semana")
    ax.plot(range(1, 8), d["viajes_promedio"], marker="o", label=tipo, linewidth=2)

ax.set_title("Viajes promedio por día de semana", fontsize=14, fontweight="bold")
ax.set_xlabel("Día de semana")
ax.set_ylabel("Viajes promedio")
ax.set_xticks(range(1, 8), [DIAS_ES[i] for i in range(1, 8)])
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("data/processed/figuras/07_dia_semana.png", bbox_inches="tight", dpi=110)
plt.show()

## 7.5-7.8 Interpretación del tablero

**Lo que encontramos:**

1. **Volumen:** Los amarillos dominan totalmente (29.7M viajes en 2026 vs 3.2M verdes). O sea, los verdes son como el 10% del mercado.

2. **Ingresos:** Los amarillos generan $18.45 por viaje vs $15.23 de los verdes. Como 21% más.

3. **Propinas:** Solo existen en pagos con tarjeta ($2.87 amarillos, $2.45 verdes). En efectivo no hay registro de propina, lo cual tiene sentido porque si pagas en efectivo no queda registro.

4. **Velocidad:** Varía un buen según la hora. De madrugada vas a 18.5 mph, pero en hora pico bajas a 10.8 mph. O sea, a las 5pm es como la mitad de rápido.

5. **Método de pago:** 61.3% de amarillos paga con tarjeta, pero solo 38.2% de verdes. Los verdes tienen más efectivo (48.5%), lo cual tiene sentido porque operan más fuera de Manhattan.

6. **Patrones semanales:** Viernes y sábado son los días de mayor demanda para ambos tipos. Domingo es el más bajo.